# Model Diagnostics: Scenario-Based Analysis

This notebook analyzes a trained Deep CFR model using fixed, realistic poker scenarios.

**Key Info:**
- D0 = HIGHEST rank card (e.g., Ace)
- D1 = Middle rank card
- D2 = LOWEST rank card (e.g., 2)
- Good poker: Keep high cards → D2 should have highest discard probability

## Configuration - Change Model Path Here

In [1]:
# ============================================
# CHANGE THIS TO YOUR MODEL PATH
# ============================================
MODEL_PATH = "../output/models/deep_cfr_hulh_scale_iter_1.pt"

# Optional: Compare with another model
COMPARE_MODEL_PATH = None  # Set to path like "../output/models/other_model.pt" to compare
# ============================================

## Setup

In [2]:
import sys
import os
import torch
import torch.nn.functional as F
import pandas as pd
from IPython.display import display, HTML

# Add paths
sys.path.insert(0, '.')
sys.path.insert(0, '..')

from network.model import DeepCFRModule
from utils.infoset_parser import parse_infoset_to_network_input

# Network output order: 9 actions
# [DISCARD_0, DISCARD_1, DISCARD_2, CHECK, CALL, FOLD, RAISE_SMALL, RAISE_MEDIUM, RAISE_LARGE]
ACTION_NAMES = ['DISCARD_0', 'DISCARD_1', 'DISCARD_2', 'CHECK', 'CALL', 'FOLD', 'RAISE_S', 'RAISE_M', 'RAISE_L']

# Action encoding in infoset strings (7 features per action):
# 'X' = check, 'C' = call, 'F' = fold, 'D' = discard
# 'r' = raise small, 'R' = raise medium, 'B' = raise large (Big)
print("Setup complete!")
print("Action encoding: X=check, C=call, F=fold, D=discard, r=raise_small, R=raise_medium, B=raise_large")

Setup complete!
Action encoding: X=check, C=call, F=fold, D=discard, r=raise_small, R=raise_medium, B=raise_large


## Define Scenarios

In [3]:
# Hardcoded realistic scenarios for each game phase
# Format: "S{street}|H:{hand}|B:{board}|A:{history}"
# Hand sorted descending by rank: D0=highest, D2=lowest
#
# Action encoding (7 features per action):
# 'X' = check, 'C' = call, 'F' = fold, 'D' = discard
# 'r' = raise small (<0.5 pot), 'R' = raise medium (0.5-1.0 pot), 'B' = raise large (>1.0 pot)

SCENARIOS = {
    # ==================== CLASSIC POKER LOGIC TESTS ====================
    # These are the "sanity check" scenarios - if the model doesn't get these right,
    # something is fundamentally wrong with the training.
    'CLASSIC COMPARISONS': {
        'legal_indices': [3, 4, 5, 6, 7, 8],  # CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L
        'scenarios': [
            # AA vs 72 - the classic comparison
            ('AA-x preflop (BEST)', 'S0|H:14s0,14s1,8s2|B:|A:'),
            ('72-x preflop (WORST)', 'S0|H:7s0,5s1,2s2|B:|A:'),
            # AA vs 72 facing a raise - should we call/raise or fold?
            ('AA-x facing small raise', 'S0|H:14s0,14s1,8s2|B:|A:r'),
            ('72-x facing small raise', 'S0|H:7s0,5s1,2s2|B:|A:r'),
            ('AA-x facing medium raise', 'S0|H:14s0,14s1,8s2|B:|A:R'),
            ('72-x facing medium raise', 'S0|H:7s0,5s1,2s2|B:|A:R'),
            ('AA-x facing big raise', 'S0|H:14s0,14s1,8s2|B:|A:B'),
            ('72-x facing big raise', 'S0|H:7s0,5s1,2s2|B:|A:B'),
            # KK vs 32 
            ('KK-x preflop', 'S0|H:13s0,13s1,6s2|B:|A:'),
            ('32-x preflop', 'S0|H:5s0,3s1,2s2|B:|A:'),
        ]
    },
    
    'PREFLOP BETTING': {
        'legal_indices': [3, 4, 5, 6, 7, 8],
        'scenarios': [
            ('Premium pair (AA-x)', 'S0|H:14s0,14s1,8s2|B:|A:'),
            ('High pair (KK-x)', 'S0|H:13s0,13s1,5s2|B:|A:'),
            ('Broadway (AKQ)', 'S0|H:14s0,13s0,12s0|B:|A:'),
            ('Medium pair (99-x)', 'S0|H:9s0,9s1,6s2|B:|A:'),
            ('Connected (JT9)', 'S0|H:11s0,10s0,9s0|B:|A:'),
            ('One high (A-7-3)', 'S0|H:14s0,7s1,3s2|B:|A:'),
            ('Low cards (7-5-3)', 'S0|H:7s0,5s1,3s2|B:|A:'),
            ('Garbage (8-4-2)', 'S0|H:8s0,4s1,2s2|B:|A:'),
            # Facing different raise sizes
            ('Premium facing small raise', 'S0|H:14s0,14s1,10s2|B:|A:r'),  # r = small raise
            ('Premium facing medium raise', 'S0|H:14s0,14s1,10s2|B:|A:R'),  # R = medium raise
            ('Premium facing big raise', 'S0|H:14s0,14s1,10s2|B:|A:B'),  # B = large raise
            ('Weak facing small raise', 'S0|H:7s0,5s1,2s2|B:|A:r'),
            ('Weak facing medium raise', 'S0|H:7s0,5s1,2s2|B:|A:R'),
        ]
    },
    
    'DISCARD ROUND': {
        'legal_indices': [0, 1, 2],
        'scenarios': [
            # CX = call (SB completes), check (BB checks) -> now at discard
            ('Clear low (A-K-3, board 9-7)', 'S0|H:14s0,13s0,3s1|B:9s2,7s2|A:CX'),
            ('Clear low (Q-J-2, board T-8)', 'S0|H:12s0,11s0,2s1|B:10s2,8s2|A:CX'),
            ('Pair + kicker (K-K-5, board A-9)', 'S0|H:13s0,13s1,5s2|B:14s2,9s2|A:CX'),
            ('Pair + kicker (9-9-3, board J-7)', 'S0|H:9s0,9s1,3s2|B:11s2,7s2|A:CX'),
            ('Flush draw (A-T-4 suited, board K-7)', 'S0|H:14s0,10s0,4s0|B:13s0,7s1|A:CX'),
            ('Straight draw (J-T-5, board 9-8)', 'S0|H:11s0,10s0,5s1|B:9s2,8s2|A:CX'),
            ('Close decision (K-Q-J, board A-9)', 'S0|H:13s0,12s0,11s0|B:14s1,9s1|A:CX'),
            ('Close decision (8-7-6, board T-5)', 'S0|H:8s0,7s0,6s0|B:10s1,5s1|A:CX'),
            # After raises with different sizes
            ('After small raise', 'S0|H:14s0,13s0,2s1|B:9s2,7s2|A:rC'),
            ('After medium raise', 'S0|H:14s0,13s0,2s1|B:9s2,7s2|A:RC'),
        ]
    },
    
    'FLOP BETTING': {
        'legal_indices': [3, 4, 5, 6, 7, 8],
        'scenarios': [
            # CXDD = call, check, both discarded -> now at flop
            ('Top pair (K-Q, board K-9-7-5)', 'S1|H:13s0,12s0|B:13s1,9s2,7s2,5s2|A:CXDD'),
            ('Two pair (K-9, board K-9-7-5)', 'S1|H:13s0,9s0|B:13s1,9s1,7s2,5s2|A:CXDD'),
            ('Set (9-9, board K-9-7-5)', 'S1|H:9s0,9s1|B:13s2,9s2,7s2,5s2|A:CXDD'),
            ('Flush draw', 'S1|H:14s0,10s0|B:13s0,7s0,9s1,5s1|A:CXDD'),
            ('Open-ended (J-T, board 9-8-K-3)', 'S1|H:11s0,10s0|B:9s1,8s1,13s2,3s2|A:CXDD'),
            ('Missed (A-Q, board K-9-7-5)', 'S1|H:14s0,12s0|B:13s1,9s2,7s2,5s2|A:CXDD'),
            ('Bottom pair (5-4, board K-9-7-5)', 'S1|H:5s0,4s0|B:13s1,9s2,7s2,5s1|A:CXDD'),
            # Facing different bet sizes
            ('Top pair facing small bet', 'S1|H:13s0,12s0|B:13s1,9s2,7s2,5s2|A:CXDDr'),  # r = small bet
            ('Top pair facing medium bet', 'S1|H:13s0,12s0|B:13s1,9s2,7s2,5s2|A:CXDDR'),  # R = medium bet
            ('Top pair facing big bet', 'S1|H:13s0,12s0|B:13s1,9s2,7s2,5s2|A:CXDDB'),  # B = large bet
            ('Air facing small bet', 'S1|H:6s0,4s0|B:13s1,9s2,7s2,5s2|A:CXDDr'),
            ('Air facing big bet', 'S1|H:6s0,4s0|B:13s1,9s2,7s2,5s2|A:CXDDB'),
        ]
    },
    
    'TURN BETTING': {
        'legal_indices': [3, 4, 5, 6, 7, 8],
        'scenarios': [
            # CXDDXX = preflop completed, discards, flop checks -> now at turn
            ('Top pair good kicker', 'S2|H:14s0,13s0|B:14s1,9s2,7s2,5s2,3s2|A:CXDDXX'),
            ('Two pair', 'S2|H:14s0,9s0|B:14s1,9s1,7s2,5s2,3s2|A:CXDDXX'),
            ('Set', 'S2|H:9s0,9s1|B:14s2,9s2,7s2,5s2,3s2|A:CXDDXX'),
            ('Flush draw turn', 'S2|H:14s0,10s0|B:13s0,7s0,9s1,5s1,2s1|A:CXDDXX'),
            ('Missed draw', 'S2|H:11s0,10s0|B:9s1,8s1,14s2,3s2,2s2|A:CXDDXX'),
            ('Second pair', 'S2|H:9s0,8s0|B:14s1,9s1,7s2,5s2,3s2|A:CXDDXX'),
            ('Weak pair', 'S2|H:5s0,4s0|B:14s1,9s2,7s2,5s1,3s2|A:CXDDXX'),
            # Raised pots with different sizes
            ('Strong after small raise', 'S2|H:14s0,14s1|B:14s2,9s2,7s2,5s2,3s2|A:rCDDXX'),
            ('Strong after medium raise', 'S2|H:14s0,14s1|B:14s2,9s2,7s2,5s2,3s2|A:RCDDXX'),
            ('Strong after big raise', 'S2|H:14s0,14s1|B:14s2,9s2,7s2,5s2,3s2|A:BCDDXX'),
            # Facing bets
            ('Bluff catcher facing small bet', 'S2|H:9s0,8s0|B:14s1,13s1,12s1,5s2,3s2|A:CXDDXXr'),
            ('Bluff catcher facing big bet', 'S2|H:9s0,8s0|B:14s1,13s1,12s1,5s2,3s2|A:CXDDXXB'),
        ]
    },
    
    'RIVER BETTING': {
        'legal_indices': [3, 4, 5, 6, 7, 8],
        'scenarios': [
            # CXDDXXXX = full runout with checks
            ('Top pair river', 'S3|H:14s0,13s0|B:14s1,9s2,7s2,5s2,3s2,2s2|A:CXDDXXXX'),
            ('Two pair river', 'S3|H:14s0,9s0|B:14s1,9s1,7s2,5s2,3s2,2s2|A:CXDDXXXX'),
            ('Set river', 'S3|H:9s0,9s1|B:14s2,9s2,7s2,5s2,3s2,2s2|A:CXDDXXXX'),
            ('Straight river', 'S3|H:11s0,10s0|B:9s1,8s1,7s2,5s2,3s2,2s2|A:CXDDXXXX'),
            ('Missed draw river', 'S3|H:14s0,10s0|B:13s1,7s1,9s2,5s2,3s2,2s2|A:CXDDXXXX'),
            ('Total air river', 'S3|H:6s0,4s0|B:14s1,13s1,12s1,9s2,7s2,2s2|A:CXDDXXXX'),
            # Facing different bet sizes
            ('Medium pair vs small bet', 'S3|H:9s0,8s0|B:14s1,13s1,12s1,5s2,3s2,2s2|A:CXDDXXXXr'),
            ('Medium pair vs medium bet', 'S3|H:9s0,8s0|B:14s1,13s1,12s1,5s2,3s2,2s2|A:CXDDXXXXR'),
            ('Medium pair vs big bet', 'S3|H:9s0,8s0|B:14s1,13s1,12s1,5s2,3s2,2s2|A:CXDDXXXXB'),
            ('Weak pair vs small bet', 'S3|H:5s0,4s0|B:14s1,13s1,12s1,9s2,5s1,2s2|A:CXDDXXXXr'),
            ('Weak pair vs big bet', 'S3|H:5s0,4s0|B:14s1,13s1,12s1,9s2,5s1,2s2|A:CXDDXXXXB'),
            # Big pots
            ('Nuts in big pot', 'S3|H:14s0,14s1|B:14s2,9s2,7s2,5s2,3s2,2s2|A:BCDDXXRX'),
        ]
    },
}

print(f"Defined {sum(len(p['scenarios']) for p in SCENARIOS.values())} scenarios across {len(SCENARIOS)} phases")
print("Action encoding: X=check, C=call, F=fold, D=discard, r=raise_small, R=raise_medium, B=raise_large")

Defined 69 scenarios across 6 phases
Action encoding: X=check, C=call, F=fold, D=discard, r=raise_small, R=raise_medium, B=raise_large


## Load Model

In [4]:
def load_model(model_path):
    """Load a model and return network + metadata."""
    print(f"Loading: {model_path}")
    model_data = torch.load(model_path, map_location='cpu', weights_only=False)
    
    network_dim = model_data.get('network_dim', 256)
    iterations = model_data.get('iterations', '?')
    
    network = DeepCFRModule(nhandcards=3, nboardcards=5, n_action_history=20, nresponses=9, dim=network_dim)
    network.load_state_dict(model_data['strategy_network_state_dict'])
    network.eval()
    
    print(f"  ✓ Loaded! Dim: {network_dim}, Iterations: {iterations}")
    return network, {'dim': network_dim, 'iterations': iterations, 'path': model_path}

# Load main model
network, model_info = load_model(MODEL_PATH)

# Optionally load comparison model
compare_network, compare_info = None, None
if COMPARE_MODEL_PATH:
    compare_network, compare_info = load_model(COMPARE_MODEL_PATH)

Loading: ../output/models/deep_cfr_hulh_scale_iter_1.pt
  ✓ Loaded! Dim: 256, Iterations: 1


## Analysis Functions

In [5]:
def analyze_scenario(network, infoset_str, legal_indices):
    """Get network output for a scenario."""
    try:
        cc, ah = parse_infoset_to_network_input(infoset_str)
        
        with torch.no_grad():
            output = network(cc, ah)
        
        logits = output[0]
        legal_logits = torch.tensor([logits[i].item() for i in legal_indices])
        legal_probs = F.softmax(legal_logits, dim=0)
        
        probs_dict = {}
        for i, idx in enumerate(legal_indices):
            probs_dict[ACTION_NAMES[idx]] = legal_probs[i].item()
        
        dominant_idx = legal_indices[torch.argmax(legal_probs).item()]
        
        return {
            'probs': probs_dict,
            'dominant': ACTION_NAMES[dominant_idx],
            'dominant_prob': torch.max(legal_probs).item(),
        }
    except Exception as e:
        return {'error': str(e)}

def format_probs(probs_dict, highlight_max=True):
    """Format probabilities as a nice string with highlighting."""
    if not probs_dict:
        return "ERROR"
    
    max_key = max(probs_dict.keys(), key=lambda k: probs_dict[k])
    parts = []
    for k, v in probs_dict.items():
        short_name = k.replace('DISCARD_', 'D').replace('CHECK', 'CHK').replace('RAISE_', 'R')
        if highlight_max and k == max_key:
            parts.append(f"**{short_name}: {v:.1%}**")
        else:
            parts.append(f"{short_name}: {v:.1%}")
    return " | ".join(parts)

print("Analysis functions ready!")

Analysis functions ready!


---
# Results by Phase
---

## 0. CLASSIC POKER LOGIC (Sanity Check)

**This is the most important section!** If the model has learned anything, it should:
- Prefer raising/calling with AA vs folding
- Prefer folding with 72 vs raising
- Show different behavior for strong vs weak hands facing raises

In [6]:
phase = 'CLASSIC COMPARISONS'
phase_data = SCENARIOS[phase]

print("═" * 80)
print("  CLASSIC POKER LOGIC - AA vs 72 (The Ultimate Sanity Check)")
print("  If the model has learned ANYTHING, AA should play very differently from 72!")
print("═" * 80)
print()

results = []
for desc, infoset in phase_data['scenarios']:
    result = analyze_scenario(network, infoset, phase_data['legal_indices'])
    
    print(f"📋 {desc}")
    print(f"   Infoset: {infoset}")
    if 'error' in result:
        print(f"   ❌ Error: {result['error']}")
    else:
        print(f"   Probs: {format_probs(result['probs'])}")
        print(f"   🎯 Predicted: {result['dominant']} ({result['dominant_prob']:.1%})")
        results.append({'desc': desc, **result})
    print()

# ==================== CLASSIC COMPARISON ANALYSIS ====================
print("─" * 80)
print("📊 SANITY CHECK ANALYSIS:")
print("─" * 80)

# Find AA and 72 scenarios
aa_open = next((r for r in results if 'AA-x preflop (BEST)' in r['desc']), None)
worst_open = next((r for r in results if '72-x preflop (WORST)' in r['desc']), None)

if aa_open and worst_open:
    # Compare raise probabilities
    aa_raise = aa_open['probs'].get('RAISE_S', 0) + aa_open['probs'].get('RAISE_M', 0) + aa_open['probs'].get('RAISE_L', 0)
    worst_raise = worst_open['probs'].get('RAISE_S', 0) + worst_open['probs'].get('RAISE_M', 0) + worst_open['probs'].get('RAISE_L', 0)
    
    aa_fold = aa_open['probs'].get('FOLD', 0)
    worst_fold = worst_open['probs'].get('FOLD', 0)
    
    print(f"\n  AA-x:   Raise total = {aa_raise:.1%}, Fold = {aa_fold:.1%}")
    print(f"  72-x:   Raise total = {worst_raise:.1%}, Fold = {worst_fold:.1%}")
    print()
    
    if aa_raise > worst_raise + 0.05:
        print("  ✅ GOOD: AA raises more than 72")
    elif aa_raise > worst_raise:
        print("  ⚠️ MARGINAL: AA raises slightly more than 72 (barely learning)")
    else:
        print("  ❌ FAIL: AA doesn't raise more than 72 - model hasn't learned basic poker!")
        
    if worst_fold > aa_fold + 0.05:
        print("  ✅ GOOD: 72 folds more than AA")
    elif worst_fold > aa_fold:
        print("  ⚠️ MARGINAL: 72 folds slightly more than AA (barely learning)")
    else:
        print("  ❌ FAIL: 72 doesn't fold more than AA - model hasn't learned basic poker!")

# Check facing raises
aa_vs_raise = next((r for r in results if 'AA-x facing medium raise' in r['desc']), None)
worst_vs_raise = next((r for r in results if '72-x facing medium raise' in r['desc']), None)

if aa_vs_raise and worst_vs_raise:
    aa_continue = 1.0 - aa_vs_raise['probs'].get('FOLD', 0)
    worst_continue = 1.0 - worst_vs_raise['probs'].get('FOLD', 0)
    
    print(f"\n  AA-x facing raise: Continue = {aa_continue:.1%}")
    print(f"  72-x facing raise: Continue = {worst_continue:.1%}")
    print()
    
    if aa_continue > worst_continue + 0.10:
        print("  ✅ GOOD: AA continues more vs raises than 72")
    else:
        print("  ❌ FAIL: AA should continue more facing a raise than 72!")

════════════════════════════════════════════════════════════════════════════════
  CLASSIC POKER LOGIC - AA vs 72 (The Ultimate Sanity Check)
  If the model has learned ANYTHING, AA should play very differently from 72!
════════════════════════════════════════════════════════════════════════════════

📋 AA-x preflop (BEST)
   Infoset: S0|H:14s0,14s1,8s2|B:|A:
   Probs: CHK: 15.0% | CALL: 13.0% | **FOLD: 24.3%** | RS: 20.8% | RM: 12.0% | RL: 14.8%
   🎯 Predicted: FOLD (24.3%)

📋 72-x preflop (WORST)
   Infoset: S0|H:7s0,5s1,2s2|B:|A:
   Probs: CHK: 15.1% | CALL: 13.0% | **FOLD: 24.3%** | RS: 20.9% | RM: 12.0% | RL: 14.7%
   🎯 Predicted: FOLD (24.3%)

📋 AA-x facing small raise
   Infoset: S0|H:14s0,14s1,8s2|B:|A:r
   Probs: CHK: 15.0% | CALL: 13.0% | **FOLD: 24.4%** | RS: 20.8% | RM: 12.0% | RL: 14.8%
   🎯 Predicted: FOLD (24.4%)

📋 72-x facing small raise
   Infoset: S0|H:7s0,5s1,2s2|B:|A:r
   Probs: CHK: 15.1% | CALL: 13.1% | **FOLD: 24.3%** | RS: 20.8% | RM: 12.0% | RL: 14.7%
   🎯 Pred

## 1. PREFLOP BETTING

In [7]:
phase = 'PREFLOP BETTING'
phase_data = SCENARIOS[phase]

print(f"═" * 80)
print(f"  {phase}")
print(f"  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L")
print(f"═" * 80)
print()

results = []
for desc, infoset in phase_data['scenarios']:
    result = analyze_scenario(network, infoset, phase_data['legal_indices'])
    
    print(f"📋 {desc}")
    print(f"   Infoset: {infoset}")
    if 'error' in result:
        print(f"   ❌ Error: {result['error']}")
    else:
        print(f"   Probs: {format_probs(result['probs'])}")
        print(f"   🎯 Predicted: {result['dominant']} ({result['dominant_prob']:.1%})")
        results.append(result)
    print()

# Average
if results:
    avg = {k: sum(r['probs'][k] for r in results) / len(results) for k in results[0]['probs']}
    print(f"─" * 80)
    print(f"📊 AVERAGE: {format_probs(avg)}")

════════════════════════════════════════════════════════════════════════════════
  PREFLOP BETTING
  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L
════════════════════════════════════════════════════════════════════════════════

📋 Premium pair (AA-x)
   Infoset: S0|H:14s0,14s1,8s2|B:|A:
   Probs: CHK: 15.0% | CALL: 13.0% | **FOLD: 24.3%** | RS: 20.8% | RM: 12.0% | RL: 14.8%
   🎯 Predicted: FOLD (24.3%)

📋 High pair (KK-x)
   Infoset: S0|H:13s0,13s1,5s2|B:|A:
   Probs: CHK: 15.1% | CALL: 13.1% | **FOLD: 24.2%** | RS: 20.8% | RM: 12.0% | RL: 14.8%
   🎯 Predicted: FOLD (24.2%)

📋 Broadway (AKQ)
   Infoset: S0|H:14s0,13s0,12s0|B:|A:
   Probs: CHK: 15.0% | CALL: 13.0% | **FOLD: 24.3%** | RS: 20.9% | RM: 12.1% | RL: 14.7%
   🎯 Predicted: FOLD (24.3%)

📋 Medium pair (99-x)
   Infoset: S0|H:9s0,9s1,6s2|B:|A:
   Probs: CHK: 15.1% | CALL: 13.1% | **FOLD: 24.3%** | RS: 20.7% | RM: 12.0% | RL: 14.8%
   🎯 Predicted: FOLD (24.3%)

📋 Connected (JT9)
   Infoset: S0|H:11s0,10s0,9s0|B:|A:


📋 Low cards (7-5-3)
   Infoset: S0|H:7s0,5s1,3s2|B:|A:
   Probs: CHK: 15.1% | CALL: 13.1% | **FOLD: 24.3%** | RS: 20.8% | RM: 12.0% | RL: 14.8%
   🎯 Predicted: FOLD (24.3%)

📋 Garbage (8-4-2)
   Infoset: S0|H:8s0,4s1,2s2|B:|A:
   Probs: CHK: 15.0% | CALL: 13.0% | **FOLD: 24.3%** | RS: 20.8% | RM: 12.0% | RL: 14.8%
   🎯 Predicted: FOLD (24.3%)

📋 Premium facing small raise
   Infoset: S0|H:14s0,14s1,10s2|B:|A:r
   Probs: CHK: 15.0% | CALL: 13.1% | **FOLD: 24.3%** | RS: 20.8% | RM: 12.0% | RL: 14.8%
   🎯 Predicted: FOLD (24.3%)

📋 Premium facing medium raise
   Infoset: S0|H:14s0,14s1,10s2|B:|A:R
   Probs: CHK: 15.0% | CALL: 13.0% | **FOLD: 24.3%** | RS: 20.8% | RM: 12.0% | RL: 14.8%
   🎯 Predicted: FOLD (24.3%)

📋 Premium facing big raise
   Infoset: S0|H:14s0,14s1,10s2|B:|A:B
   Probs: CHK: 15.0% | CALL: 13.0% | **FOLD: 24.3%** | RS: 20.8% | RM: 12.1% | RL: 14.9%
   🎯 Predicted: FOLD (24.3%)

📋 Weak facing small raise
   Infoset: S0|H:7s0,5s1,2s2|B:|A:r
   Probs: CHK: 15.1% | CALL: 13.

## 2. DISCARD ROUND ⚠️ Critical Phase

In [8]:
phase = 'DISCARD ROUND'
phase_data = SCENARIOS[phase]

print(f"═" * 80)
print(f"  {phase}")
print(f"  D0 = HIGHEST card | D1 = Middle | D2 = LOWEST card")
print(f"  ⚠️ Good strategy: D2 should be highest (discard low cards)")
print(f"═" * 80)
print()

results = []
for desc, infoset in phase_data['scenarios']:
    result = analyze_scenario(network, infoset, phase_data['legal_indices'])
    
    print(f"📋 {desc}")
    print(f"   Infoset: {infoset}")
    if 'error' in result:
        print(f"   ❌ Error: {result['error']}")
    else:
        # Check if prediction makes sense
        is_good = result['dominant'] == 'DISCARD_2'
        status = "✅" if is_good else "⚠️"
        
        print(f"   Probs: {format_probs(result['probs'])}")
        print(f"   {status} Predicted: {result['dominant']} ({result['dominant_prob']:.1%})")
        results.append(result)
    print()

# Average and analysis
if results:
    avg = {k: sum(r['probs'][k] for r in results) / len(results) for k in results[0]['probs']}
    print(f"─" * 80)
    print(f"📊 AVERAGE: {format_probs(avg)}")
    print()
    
    # Analysis
    d0, d1, d2 = avg['DISCARD_0'], avg['DISCARD_1'], avg['DISCARD_2']
    if d2 > d0 and d2 > d1:
        print("✅ GOOD: Model prefers discarding low cards (D2)")
    elif d0 > d1 and d0 > d2:
        print("⚠️ WARNING: Model prefers discarding HIGH cards (D0) - this is backwards!")
    else:
        print("⚠️ UNCLEAR: Model doesn't have clear discard preference yet")

════════════════════════════════════════════════════════════════════════════════
  DISCARD ROUND
  D0 = HIGHEST card | D1 = Middle | D2 = LOWEST card
  ⚠️ Good strategy: D2 should be highest (discard low cards)
════════════════════════════════════════════════════════════════════════════════

📋 Clear low (A-K-3, board 9-7)
   Infoset: S0|H:14s0,13s0,3s1|B:9s2,7s2|A:CX
   Probs: **D0: 33.7%** | D1: 32.7% | D2: 33.6%
   ⚠️ Predicted: DISCARD_0 (33.7%)

📋 Clear low (Q-J-2, board T-8)
   Infoset: S0|H:12s0,11s0,2s1|B:10s2,8s2|A:CX
   Probs: **D0: 33.7%** | D1: 32.7% | D2: 33.6%
   ⚠️ Predicted: DISCARD_0 (33.7%)

📋 Pair + kicker (K-K-5, board A-9)
   Infoset: S0|H:13s0,13s1,5s2|B:14s2,9s2|A:CX
   Probs: **D0: 33.7%** | D1: 32.7% | D2: 33.6%
   ⚠️ Predicted: DISCARD_0 (33.7%)

📋 Pair + kicker (9-9-3, board J-7)
   Infoset: S0|H:9s0,9s1,3s2|B:11s2,7s2|A:CX
   Probs: **D0: 33.7%** | D1: 32.7% | D2: 33.6%
   ⚠️ Predicted: DISCARD_0 (33.7%)

📋 Flush draw (A-T-4 suited, board K-7)
   Infoset: S0|

## 3. FLOP BETTING

In [9]:
phase = 'FLOP BETTING'
phase_data = SCENARIOS[phase]

print(f"═" * 80)
print(f"  {phase}")
print(f"  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L")
print(f"═" * 80)
print()

results = []
for desc, infoset in phase_data['scenarios']:
    result = analyze_scenario(network, infoset, phase_data['legal_indices'])
    
    print(f"📋 {desc}")
    print(f"   Infoset: {infoset}")
    if 'error' in result:
        print(f"   ❌ Error: {result['error']}")
    else:
        print(f"   Probs: {format_probs(result['probs'])}")
        print(f"   🎯 Predicted: {result['dominant']} ({result['dominant_prob']:.1%})")
        results.append(result)
    print()

if results:
    avg = {k: sum(r['probs'][k] for r in results) / len(results) for k in results[0]['probs']}
    print(f"─" * 80)
    print(f"📊 AVERAGE: {format_probs(avg)}")

════════════════════════════════════════════════════════════════════════════════
  FLOP BETTING
  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L
════════════════════════════════════════════════════════════════════════════════

📋 Top pair (K-Q, board K-9-7-5)
   Infoset: S1|H:13s0,12s0|B:13s1,9s2,7s2,5s2|A:CXDD
   Probs: CHK: 15.0% | CALL: 13.2% | **FOLD: 24.5%** | RS: 20.5% | RM: 12.0% | RL: 14.9%
   🎯 Predicted: FOLD (24.5%)

📋 Two pair (K-9, board K-9-7-5)
   Infoset: S1|H:13s0,9s0|B:13s1,9s1,7s2,5s2|A:CXDD
   Probs: CHK: 15.0% | CALL: 13.2% | **FOLD: 24.4%** | RS: 20.5% | RM: 12.0% | RL: 14.9%
   🎯 Predicted: FOLD (24.4%)

📋 Set (9-9, board K-9-7-5)
   Infoset: S1|H:9s0,9s1|B:13s2,9s2,7s2,5s2|A:CXDD
   Probs: CHK: 15.1% | CALL: 13.2% | **FOLD: 24.5%** | RS: 20.4% | RM: 12.0% | RL: 14.9%
   🎯 Predicted: FOLD (24.5%)

📋 Flush draw
   Infoset: S1|H:14s0,10s0|B:13s0,7s0,9s1,5s1|A:CXDD
   Probs: CHK: 15.0% | CALL: 13.2% | **FOLD: 24.5%** | RS: 20.5% | RM: 12.0% | RL: 14.9%
 

📋 Air facing small bet
   Infoset: S1|H:6s0,4s0|B:13s1,9s2,7s2,5s2|A:CXDDr
   Probs: CHK: 15.0% | CALL: 13.1% | **FOLD: 24.5%** | RS: 20.5% | RM: 12.0% | RL: 14.9%
   🎯 Predicted: FOLD (24.5%)

📋 Air facing big bet
   Infoset: S1|H:6s0,4s0|B:13s1,9s2,7s2,5s2|A:CXDDB
   Probs: CHK: 15.0% | CALL: 13.1% | **FOLD: 24.5%** | RS: 20.5% | RM: 12.0% | RL: 14.9%
   🎯 Predicted: FOLD (24.5%)

────────────────────────────────────────────────────────────────────────────────
📊 AVERAGE: CHK: 15.0% | CALL: 13.2% | **FOLD: 24.5%** | RS: 20.5% | RM: 12.0% | RL: 14.9%


## 4. TURN BETTING

In [10]:
phase = 'TURN BETTING'
phase_data = SCENARIOS[phase]

print(f"═" * 80)
print(f"  {phase}")
print(f"  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L")
print(f"═" * 80)
print()

results = []
for desc, infoset in phase_data['scenarios']:
    result = analyze_scenario(network, infoset, phase_data['legal_indices'])
    
    print(f"📋 {desc}")
    print(f"   Infoset: {infoset}")
    if 'error' in result:
        print(f"   ❌ Error: {result['error']}")
    else:
        print(f"   Probs: {format_probs(result['probs'])}")
        print(f"   🎯 Predicted: {result['dominant']} ({result['dominant_prob']:.1%})")
        results.append(result)
    print()

if results:
    avg = {k: sum(r['probs'][k] for r in results) / len(results) for k in results[0]['probs']}
    print(f"─" * 80)
    print(f"📊 AVERAGE: {format_probs(avg)}")

════════════════════════════════════════════════════════════════════════════════
  TURN BETTING
  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L
════════════════════════════════════════════════════════════════════════════════

📋 Top pair good kicker
   Infoset: S2|H:14s0,13s0|B:14s1,9s2,7s2,5s2,3s2|A:CXDDXX
   Probs: CHK: 15.0% | CALL: 13.1% | **FOLD: 24.5%** | RS: 20.5% | RM: 12.0% | RL: 14.9%
   🎯 Predicted: FOLD (24.5%)

📋 Two pair
   Infoset: S2|H:14s0,9s0|B:14s1,9s1,7s2,5s2,3s2|A:CXDDXX
   Probs: CHK: 15.0% | CALL: 13.2% | **FOLD: 24.5%** | RS: 20.5% | RM: 12.0% | RL: 14.9%
   🎯 Predicted: FOLD (24.5%)

📋 Set
   Infoset: S2|H:9s0,9s1|B:14s2,9s2,7s2,5s2,3s2|A:CXDDXX
   Probs: CHK: 15.1% | CALL: 13.2% | **FOLD: 24.5%** | RS: 20.4% | RM: 12.0% | RL: 14.9%
   🎯 Predicted: FOLD (24.5%)

📋 Flush draw turn
   Infoset: S2|H:14s0,10s0|B:13s0,7s0,9s1,5s1,2s1|A:CXDDXX
   Probs: CHK: 15.0% | CALL: 13.2% | **FOLD: 24.5%** | RS: 20.5% | RM: 12.0% | RL: 14.9%
   🎯 Predicted: FOLD (2

## 5. RIVER BETTING

In [11]:
phase = 'RIVER BETTING'
phase_data = SCENARIOS[phase]

print(f"═" * 80)
print(f"  {phase}")
print(f"  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L")
print(f"═" * 80)
print()

results = []
for desc, infoset in phase_data['scenarios']:
    result = analyze_scenario(network, infoset, phase_data['legal_indices'])
    
    print(f"📋 {desc}")
    print(f"   Infoset: {infoset}")
    if 'error' in result:
        print(f"   ❌ Error: {result['error']}")
    else:
        print(f"   Probs: {format_probs(result['probs'])}")
        print(f"   🎯 Predicted: {result['dominant']} ({result['dominant_prob']:.1%})")
        results.append(result)
    print()

if results:
    avg = {k: sum(r['probs'][k] for r in results) / len(results) for k in results[0]['probs']}
    print(f"─" * 80)
    print(f"📊 AVERAGE: {format_probs(avg)}")

════════════════════════════════════════════════════════════════════════════════
  RIVER BETTING
  Legal actions: CHECK, CALL, FOLD, RAISE_S, RAISE_M, RAISE_L
════════════════════════════════════════════════════════════════════════════════

📋 Top pair river
   Infoset: S3|H:14s0,13s0|B:14s1,9s2,7s2,5s2,3s2,2s2|A:CXDDXXXX
   Probs: CHK: 15.0% | CALL: 13.1% | **FOLD: 24.5%** | RS: 20.5% | RM: 12.0% | RL: 14.9%
   🎯 Predicted: FOLD (24.5%)

📋 Two pair river
   Infoset: S3|H:14s0,9s0|B:14s1,9s1,7s2,5s2,3s2,2s2|A:CXDDXXXX
   Probs: CHK: 15.0% | CALL: 13.1% | **FOLD: 24.5%** | RS: 20.5% | RM: 12.0% | RL: 14.9%
   🎯 Predicted: FOLD (24.5%)

📋 Set river
   Infoset: S3|H:9s0,9s1|B:14s2,9s2,7s2,5s2,3s2,2s2|A:CXDDXXXX
   Probs: CHK: 15.1% | CALL: 13.2% | **FOLD: 24.5%** | RS: 20.4% | RM: 12.0% | RL: 14.9%
   🎯 Predicted: FOLD (24.5%)

📋 Straight river
   Infoset: S3|H:11s0,10s0|B:9s1,8s1,7s2,5s2,3s2,2s2|A:CXDDXXXX
   Probs: CHK: 15.0% | CALL: 13.2% | **FOLD: 24.5%** | RS: 20.5% | RM: 12.0% | RL: 

---
# Summary Table
---

In [12]:
print("═" * 80)
print(f"  SUMMARY: {os.path.basename(MODEL_PATH)} ({model_info['iterations']} iterations)")
print("═" * 80)
print()

summary_data = []

for phase_name, phase_data in SCENARIOS.items():
    results = []
    for desc, infoset in phase_data['scenarios']:
        result = analyze_scenario(network, infoset, phase_data['legal_indices'])
        if 'probs' in result:
            results.append(result)
    
    if results:
        avg = {k: sum(r['probs'][k] for r in results) / len(results) for k in results[0]['probs']}
        dominant = max(avg.keys(), key=lambda k: avg[k])
        
        row = {'Phase': phase_name, 'Dominant': dominant, 'Prob': f"{avg[dominant]:.1%}"}
        for k, v in avg.items():
            row[k] = f"{v:.1%}"
        summary_data.append(row)

# Display as DataFrame
df = pd.DataFrame(summary_data)
display(df)

════════════════════════════════════════════════════════════════════════════════
  SUMMARY: deep_cfr_hulh_scale_iter_1.pt (1 iterations)
════════════════════════════════════════════════════════════════════════════════



,Phase,Dominant,Prob,CHECK,CALL,FOLD,RAISE_S,RAISE_M,RAISE_L,DISCARD_0,DISCARD_1,DISCARD_2
0,CLASSIC COMPARISONS,FOLD,24.3%,15.0%,13.0%,24.3%,20.8%,12.0%,14.8%,NaN,NaN,NaN
1,PREFLOP BETTING,FOLD,24.3%,15.0%,13.0%,24.3%,20.8%,12.0%,14.8%,NaN,NaN,NaN
2,DISCARD ROUND,DISCARD_0,33.7%,NaN,NaN,NaN,NaN,NaN,NaN,33.7%,32.8%,33.6%
3,FLOP BETTING,FOLD,24.5%,15.0%,13.2%,24.5%,20.5%,12.0%,14.9%,NaN,NaN,NaN
4,TURN BETTING,FOLD,24.5%,15.1%,13.2%,24.5%,20.4%,12.0%,14.9%,NaN,NaN,NaN
5,RIVER BETTING,FOLD,24.5%,15.0%,13.1%,24.5%,20.5%,12.0%,14.9%,NaN,NaN,NaN


---
# Model Comparison (if enabled)
---

In [13]:
if compare_network is not None:
    print("═" * 80)
    print("  MODEL COMPARISON")
    print("═" * 80)
    print(f"  Model 1: {os.path.basename(MODEL_PATH)} ({model_info['iterations']} iters)")
    print(f"  Model 2: {os.path.basename(COMPARE_MODEL_PATH)} ({compare_info['iterations']} iters)")
    print()
    
    comparison_data = []
    
    for phase_name, phase_data in SCENARIOS.items():
        # Model 1
        results1 = [analyze_scenario(network, infoset, phase_data['legal_indices']) 
                    for _, infoset in phase_data['scenarios']]
        results1 = [r for r in results1 if 'probs' in r]
        
        # Model 2
        results2 = [analyze_scenario(compare_network, infoset, phase_data['legal_indices']) 
                    for _, infoset in phase_data['scenarios']]
        results2 = [r for r in results2 if 'probs' in r]
        
        if results1 and results2:
            avg1 = {k: sum(r['probs'][k] for r in results1) / len(results1) for k in results1[0]['probs']}
            avg2 = {k: sum(r['probs'][k] for r in results2) / len(results2) for k in results2[0]['probs']}
            
            dom1 = max(avg1.keys(), key=lambda k: avg1[k])
            dom2 = max(avg2.keys(), key=lambda k: avg2[k])
            
            comparison_data.append({
                'Phase': phase_name,
                'Model1 Dominant': f"{dom1} ({avg1[dom1]:.1%})",
                'Model2 Dominant': f"{dom2} ({avg2[dom2]:.1%})",
            })
    
    df = pd.DataFrame(comparison_data)
    display(df)
else:
    print("ℹ️ Set COMPARE_MODEL_PATH at the top to compare two models")

ℹ️ Set COMPARE_MODEL_PATH at the top to compare two models
